# Date Attribution and Dissolve Workflow for EDO Vector Outputs

This notebook prepares drought-related vector datasets exported from ArcGIS processing workflows.

The workflow:
- connects to a file geodatabase,
- lists available feature classes,
- selects feature classes containing `final` in their name,
- extracts acquisition dates from feature class names,
- writes the extracted date into a new attribute field,
- exports the datasets as shapefiles,
- and dissolves the exported shapefiles by `gridcode` and `date`.

The notebook is part of a GIS thesis workflow for drought analysis in Hungary.

## ArcPy Environment Setup

This notebook requires an **ArcGIS Pro Python environment** because it uses the `arcpy` library.

Before running:
- switch the Jupyter kernel to an ArcGIS Pro environment (e.g. `arcgispro-py3`)
- or run the notebook directly inside ArcGIS Pro

Standard Python environments (e.g. Anaconda) will not support ArcPy.


## Define Workspace and Paths

This section sets:
- the input geodatabase workspace,
- and the output folder for processed files.

Relative paths are used for portability and should be adjusted based on your local project structure.

In [ ]:
import arcpy
import os
import re
from datetime import datetime

# --- Paths ---
gdb = r"./Results_copy.gdb"
out_folder = r"./Data/EuropeanDroughtObservatory/Data_dated_py"

arcpy.env.workspace = gdb
arcpy.env.overwriteOutput = True

## List Feature Classes

This step retrieves all feature classes from the geodatabase workspace.

It provides an overview of available datasets before further processing.

In [ ]:
fcs = arcpy.ListFeatureClasses()

print("total features:", len(fcs))

for fc in fcs:
    print(fc)

In [ ]:
for fc in fcs:

    if "final" not in fc.lower():
        continue

    print("Feldolgozás:", fc)

    match = re.search(r'(\d{8})', fc)
    if not match:
        continue

    date_str = match.group(1)
    date_obj = datetime.strptime(date_str, "%Y%m%d")

    # date mező
    fields = [f.name for f in arcpy.ListFields(fc)]

    if "date" not in fields:
        arcpy.AddField_management(fc, "date", "DATE")

    expr = f"datetime({date_obj.year}, {date_obj.month}, {date_obj.day})"

    arcpy.CalculateField_management(
        fc,
        "date",
        expr,
        "PYTHON3"
        )

    # SHP export
    out_name = f"EDO_Hun_{date_str}.shp"
    out_path = os.path.join(out_folder, out_name)

    arcpy.CopyFeatures_management(fc, out_path)

    print("  -> export kész:", out_name)

print("Files ready.")

In [ ]:
import glob

shps = glob.glob(os.path.join(out_folder, "EDO_Hun_*.shp"))

for shp in shps:

    print("Dissolve:", shp)

    out_name = shp.replace(".shp", "_diss.shp")
    out_path = os.path.join(out_folder, out_name)

    arcpy.Dissolve_management(
        shp,
        out_path,
        ["gridcode", "date"]
    )

    print("  -> kész:", out_name)

print("Dissolve done.")